# Week 1 homework: the same workflow, a different slice of the country

**Due:** Monday of Week 2, 11:59 pm Central. **Points:** 10. **Estimated time:** 1.5–2 hours (Week 1 is a setup week, so this homework is short).

**Submit** the completed `.ipynb` with every cell executed, via Canvas.

Two reminders from the syllabus: the cells marked *Write this yourself* must be your own work, and the last cell asks you to declare any AI-assistant use.

> **Data card.** *CAMELS catchment attributes (version 2.0).* Source: Addor et al. (2017); attribute files `camels_attributes_v2.0` (DOI 10.5065/D6G73C3Q), Zenodo record 15529996 (record version 1.2, which also carries the time series, DOI 10.5065/D6MW2F4D). License: CC BY 4.0; redistribution permitted with attribution. What: static attributes for 671 US catchments; features = the 27 attributes of Kratzert et al. (2019), with units as tabulated in the Week 1 lab; target = `q_mean`, mean daily streamflow in mm/day. Size: 0.6 MB, seven text files. Coverage: CONUS; climate and flow attributes are water-year 1990–2009 means, vegetation 2002–2014, soils and geology static. Pre-processing by instructor: none. Citation: Addor et al. (2017), *HESS* 21, 5293–5313; Kratzert et al. (2019), *HESS* 23, 5089–5110.


In [ ]:
# --- Setup: all imports, the seed, plot settings. The same block opens every notebook in this course. ---
import os
import random
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

seed = 42                      # one seed for the whole notebook; re-set it before any cell you re-run
random.seed(seed)
np.random.seed(seed)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["font.size"] = 12
# Okabe–Ito: eight colorblind-safe colors as the default color cycle
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#E69F00", "#56B4E9", "#009E73", "#F0E442",
                                                     "#0072B2", "#D55E00", "#CC79A7", "#000000"])


## Provided: get and prepare the data

> **AI assistant: good use here.** These cells are given to you. Read them, run them, check the shapes.


In [ ]:
data_dir = "data"                       # created next to this notebook; ignored by git
os.makedirs(data_dir, exist_ok=True)

base_url = "https://zenodo.org/records/15529996/files/"
attribute_files = ["camels_clim", "camels_geol", "camels_hydro", "camels_name",
                   "camels_soil", "camels_topo", "camels_vege"]

for name in attribute_files:
    fname = os.path.join(data_dir, name + ".txt")
    if not os.path.exists(fname):       # download once; a hand-placed file is simply used
        print("downloading", name)
        urllib.request.urlretrieve(base_url + name + ".txt?download=1", fname)
print("all seven attribute files are in", data_dir)


In [ ]:
# The seven files share the column gauge_id. Read each one and join them side by side.
tables = []
for name in attribute_files:
    table = pd.read_csv(os.path.join(data_dir, name + ".txt"), sep=";", dtype={"gauge_id": str})
    tables.append(table.set_index("gauge_id"))
camels = pd.concat(tables, axis=1)      # one row per catchment, every attribute as a column
print(camels.shape)                     # (n_catchments, n_columns)


In [ ]:
# The 27 static attributes used by Kratzert et al. (2019). None is computed from streamflow, which matters:
# anything derived from the target (q_mean, runoff_ratio, baseflow_index, ...) would be cheating.
# (One subtlety: CAMELS' potential evaporation uses a Priestley-Taylor coefficient calibrated per catchment inside
# a rainfall-runoff model (Newman et al., 2015), so pet_mean and aridity are not purely meteorological.)
feature_names = [
    "p_mean", "pet_mean", "aridity", "p_seasonality", "frac_snow",
    "high_prec_freq", "high_prec_dur", "low_prec_freq", "low_prec_dur",
    "elev_mean", "slope_mean", "area_gages2",
    "frac_forest", "lai_max", "lai_diff", "gvf_max", "gvf_diff",
    "soil_depth_pelletier", "soil_depth_statsgo", "soil_porosity", "soil_conductivity",
    "max_water_content", "sand_frac", "silt_frac", "clay_frac",
    "carbonate_rocks_frac", "geol_permeability",
]
target_name = "q_mean"                  # mean daily streamflow, mm/day

catchments = camels.dropna(subset=feature_names + [target_name])

def rmse(y: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y - y_pred) ** 2)))

print(len(catchments), "catchments with every feature and the target")


## Question 1 (4 points): east and west

The lab fitted one model to the whole country. Split the catchments into an **eastern** group (gauge longitude east of −100°, `gauge_lon > -100`; 433 catchments) and a **western** group (the other 237), and for each group repeat the lab's pipeline: a 75/25 train/test split with `random_state=seed`, the mean-of-training-set baseline, and a 27-feature linear regression on standardized features. Report, for each group, the baseline test RMSE and the model test RMSE in mm/day; the provided print line expects them in `baseline_rmse` and `model_rmse`.

Deliverable: four printed numbers with units, labeled east/west and baseline/model.

> **Write this yourself.**

### Answer


In [ ]:
east = catchments[catchments["gauge_lon"] > -100]
west = catchments[catchments["gauge_lon"] <= -100]
print(len(east), "eastern catchments,", len(west), "western catchments")

results = {}
for region_name, region in [("east", east), ("west", west)]:
    X = region[feature_names].values
    y = region[target_name].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    # YOUR CODE HERE
    results[region_name] = (baseline_rmse, model_rmse)
    print(f"{region_name}: baseline test RMSE = {baseline_rmse:.3f} mm/day,  27-feature model test RMSE = {model_rmse:.3f} mm/day")


In [ ]:
# Self-check: run this cell. It prints only the last line if all is well.
assert set(results) == {"east", "west"}, "results should have an 'east' and a 'west' entry"
for region_name, (baseline_rmse, model_rmse) in results.items():
    assert 0 < model_rmse < baseline_rmse, f"{region_name}: the model should beat the baseline"
print("All checks passed.")


## Question 2 (4 points): the best single attribute

In the lab, precipitation alone removed most of the baseline's error. Is it the best single predictor? For each of `p_mean`, `aridity`, `pet_mean`, `elev_mean` and `frac_snow`, fit a one-feature linear regression on the whole-country training set (75/25 split, `random_state=seed`) and report the test RMSE. Then make one bar chart of the five RMSEs with labeled axes and a horizontal line for the baseline.

Deliverable: the bar chart (3 points), plus one sentence naming the best single attribute (1 point).

> **Write this yourself.**

### Answer


In [ ]:
X = catchments[feature_names].values
y = catchments[target_name].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
baseline_rmse = rmse(y_test, np.full_like(y_test, y_train.mean()))

candidates = ["p_mean", "aridity", "pet_mean", "elev_mean", "frac_snow"]
single_feature_rmse = {}
# YOUR CODE HERE
for name, value in single_feature_rmse.items():
    print(f"{name:10s} test RMSE = {value:.3f} mm/day")


In [ ]:
# Self-check
assert set(single_feature_rmse) == set(candidates), "one RMSE per candidate attribute"
assert all(v > 0 for v in single_feature_rmse.values())
assert single_feature_rmse["p_mean"] < baseline_rmse, "p_mean alone should beat the baseline"
print("All checks passed.")


*Your interpretation here.*


## Question 3 (2 points): interpret

In two or three sentences, compare the two groups of Question 1 in two ways: by the model's absolute test RMSE, and by the model's test RMSE as a fraction of each group's baseline RMSE. Which comparison says the model "works better" in one group, and which group is it? Then give one physical reason the absolute errors differ.

> **Write this yourself.**

### Answer

*Your interpretation here.*


## AI-use declaration

The syllabus requires this declaration on every homework. Fill in one row per tool you used (add rows as needed); write "none" in the first row if you used no assistant.

| Tool | Cells (question numbers) | What you asked it for | What you checked |
|---|---|---|---|
| | | | |
